# Phase 4: Baseline Machine Learning Model
## Fraud Detection System

This notebook establishes benchmark performance metrics using **Logistic Regression** as our baseline model.
We evaluate two baseline experiments on held-out test data:
1. **Experiment A:** Unweighted Logistic Regression (`class_weight=None`)
2. **Experiment B:** Class-Weighted Logistic Regression (`class_weight='balanced'`)

The goal is to understand baseline classifier performance, measure the trade-off between Precision and Recall under severe class imbalance, and set performance targets for more complex non-linear models in Phase 5.

--- 
## 1. Imports
Importing project modules (`src.data.preprocessing`, `src.models.baseline`, `src.models.evaluate`) and analytical libraries.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Add project root to sys.path
sys.path.append('..')

from src.data.preprocessing import process_raw_data
from src.models.baseline import train_baseline_model
from src.models.evaluate import evaluate_model, compare_evaluation_results

# Set plot styles
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

--- 
## 2. Load & Preprocess Data
Reusing the leakage-free preprocessing pipeline from Phase 3 (`src/data/preprocessing.py`).
Steps performed:
1. Deduplication (removing 1,081 identical records).
2. Stratified 80/20 train/test split.
3. RobustScaler fitting on `Time` and `Amount` (training data only).

In [ ]:
data_path = os.path.join('..', 'data', 'raw', 'creditcard.csv') if os.path.exists(os.path.join('..', 'data', 'raw', 'creditcard.csv')) else os.path.join('data', 'raw', 'creditcard.csv')
raw_df = pd.read_csv(data_path)

# Process data with deduplication and stratified split
processed_data = process_raw_data(raw_df, target_col='Class', remove_duplicates=True, test_size=0.2, random_state=42)

X_train = processed_data['X_train_scaled']
X_test = processed_data['X_test_scaled']
y_train = processed_data['y_train']
y_test = processed_data['y_test']

print(f"Training set shape: {X_train.shape}, Target distribution: {y_train.value_counts().to_dict()}")
print(f"Test set shape:     {X_test.shape}, Target distribution: {y_test.value_counts().to_dict()}")

--- 
## 3. Experiment A — Unweighted Logistic Regression
Training baseline `LogisticRegression` with `class_weight=None`.

In [ ]:
print("Training Experiment A: Unweighted Logistic Regression...")
pipeline_unweighted = train_baseline_model(
    X_train, y_train, class_weight=None, solver='lbfgs', max_iter=1000, random_state=42
)

# Evaluate on test data
results_unweighted = evaluate_model(pipeline_unweighted, X_test, y_test, model_name='Logistic Regression (Unweighted)')
print("Experiment A Evaluation Complete.")

--- 
## 4. Experiment B — Class-Weighted Logistic Regression
Training baseline `LogisticRegression` with `class_weight='balanced'`.

In [ ]:
print("Training Experiment B: Class-Weighted ('balanced') Logistic Regression...")
pipeline_balanced = train_baseline_model(
    X_train, y_train, class_weight='balanced', solver='lbfgs', max_iter=1000, random_state=42
)

# Evaluate on test data
results_balanced = evaluate_model(pipeline_balanced, X_test, y_test, model_name='Logistic Regression (Balanced)')
print("Experiment B Evaluation Complete.")

--- 
## 5. Quantitative Model Comparison
Comparing Precision, Recall, F1-Score, PR-AUC, ROC-AUC, and Confusion Matrix breakdown across both baseline experiments.

In [ ]:
comparison_df = compare_evaluation_results([results_unweighted, results_balanced])
print("=== Baseline Performance Comparison Table ===")
display(comparison_df)

--- 
## 6. Confusion Matrix Analysis
Visualizing confusion matrices for both baseline models.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Unweighted Confusion Matrix
sns.heatmap(results_unweighted['confusion_matrix'], annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Legit (0)', 'Fraud (1)'], yticklabels=['Legit (0)', 'Fraud (1)'])
axes[0].set_title('Unweighted Logistic Regression\n(TP=56, FN=39, FP=9, TN=56642)')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Balanced Confusion Matrix
sns.heatmap(results_balanced['confusion_matrix'], annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=['Legit (0)', 'Fraud (1)'], yticklabels=['Legit (0)', 'Fraud (1)'])
axes[1].set_title('Class-Weighted Logistic Regression\n(TP=83, FN=12, FP=1393, TN=55258)')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.show()

--- 
## 7. Precision-Recall (PR) and ROC Curves
Plotting PR and ROC curves using predicted positive class probabilities.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Precision-Recall Curves
ax1.plot(results_unweighted['pr_curve']['recall'], results_unweighted['pr_curve']['precision'],
         label=f"Unweighted (PR-AUC = {results_unweighted['pr_auc']:.4f})", color='blue', linewidth=2)
ax1.plot(results_balanced['pr_curve']['recall'], results_balanced['pr_curve']['precision'],
         label=f"Balanced (PR-AUC = {results_balanced['pr_auc']:.4f})", color='orange', linewidth=2)
ax1.set_title('Precision-Recall Curve (Primary Metric)')
ax1.set_xlabel('Recall (Sensitivity)')
ax1.set_ylabel('Precision')
ax1.legend(loc='lower left')

# ROC Curves
ax2.plot(results_unweighted['roc_curve']['fpr'], results_unweighted['roc_curve']['tpr'],
         label=f"Unweighted (ROC-AUC = {results_unweighted['roc_auc']:.4f})", color='blue', linewidth=2)
ax2.plot(results_balanced['roc_curve']['fpr'], results_balanced['roc_curve']['tpr'],
         label=f"Balanced (ROC-AUC = {results_balanced['roc_auc']:.4f})", color='orange', linewidth=2)
ax2.plot([0, 1], [0, 1], 'k--', label='Random Chance')
ax2.set_title('ROC Curve')
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.legend(loc='lower right')

plt.tight_layout()
plt.show()

--- 
## 8. Save Serialized Baseline Model Artifacts
Saving the complete preprocessor + model pipelines to `models/`.

In [ ]:
models_dir = os.path.join('..', 'models') if os.path.exists(os.path.join('..', 'models')) else 'models'
os.makedirs(models_dir, exist_ok=True)

path_unweighted = os.path.join(models_dir, 'logistic_regression_baseline.joblib')
path_balanced = os.path.join(models_dir, 'logistic_regression_balanced.joblib')

joblib.dump(pipeline_unweighted, path_unweighted)
joblib.dump(pipeline_balanced, path_balanced)

print(f"Saved unweighted baseline model to: {path_unweighted}")
print(f"Saved balanced baseline model to:   {path_balanced}")

--- 
## 9. Baseline Findings & Phase 5 Targets

### Key Findings:
1. **Unweighted Logistic Regression:**
   - **Precision:** 86.15% (Very low false alarm rate: 9 false positives).
   - **Recall:** 58.95% (Misses 39 out of 95 fraud cases).
   - **F1-Score:** 0.7000, **PR-AUC:** 0.6951, **ROC-AUC:** 0.9575.
   - **Accuracy:** 99.9154% (Misleadingly high due to majority class dominance).

2. **Class-Weighted ('balanced') Logistic Regression:**
   - **Recall:** Increases from 58.95% to **87.37%** (Detects 83 out of 95 fraud cases, missing only 12).
   - **Precision:** Drops dramatically from 86.15% to **5.62%** (Generates 1,393 false positive alarms).
   - **PR-AUC:** 0.6719, **ROC-AUC:** 0.9657.

### Modeling Implications for Phase 5:
- **Precision vs. Recall Trade-Off:** The default decision threshold (0.50) with class weighting captures significantly more fraud but floods the system with false alarms. Non-linear tree ensemble models (Random Forest, Gradient Boosting) in Phase 5 will aim to improve Recall without suffering such severe Precision penalties.